In [6]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [15]:
# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
context_length = 128 # what is the maximum context length for predictions?
max_epochs = 1000
eval_interval = 50
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 20
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
# ------------

torch.manual_seed(1337)

In [8]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

text[:1000]

"First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citizens, the patricians good.\nWhat authority surfeits on would relieve us: if they\nwould yield us but the superfluity, while it were\nwholesome, we might guess they relieved us humanely;\nbut they think we are too dear: the leanness that\nafflicts us, the object of our misery, is as an\ninventory to particularise their abundance; our\nsufferance is a gain to them Let us revenge this with\nour pikes, ere we become rakes: for the gods know I\nspeak this in hunger 

In [10]:
import tokenizer as tk

vocab_size = 300
tokenizer = tk.BPE(text, vocab_size)
tokenizer.train()

Merging (101, 32) into new token 256
Merging (116, 104) into new token 257
Merging (116, 32) into new token 258
Merging (115, 32) into new token 259
Merging (100, 32) into new token 260
Merging (44, 32) into new token 261
Merging (111, 117) into new token 262
Merging (101, 114) into new token 263
Merging (105, 110) into new token 264
Merging (121, 32) into new token 265
Merging (97, 110) into new token 266
Merging (58, 10) into new token 267
Merging (111, 114) into new token 268
Merging (111, 32) into new token 269
Merging (101, 110) into new token 270
Merging (10, 10) into new token 271
Merging (97, 114) into new token 272
Merging (32, 257) into new token 273
Merging (111, 110) into new token 274
Merging (108, 108) into new token 275
Merging (104, 97) into new token 276
Merging (44, 10) into new token 277
Merging (46, 271) into new token 278
Merging (105, 259) into new token 279
Merging (101, 115) into new token 280
Merging (121, 262) into new token 281
Merging (32, 115) into new toke

In [11]:
data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

train_data[:1000]

tensor([ 70, 299, 115, 258,  67, 105, 116, 105, 122, 270, 267,  66, 101, 102,
        268, 256, 119, 256, 112, 114, 111,  99, 101, 101, 260, 266, 265, 102,
        117, 114, 257, 263, 261, 104, 101, 272, 287, 256, 115, 112, 286, 107,
        278,  65, 275, 267,  83, 112, 286, 107, 261, 115, 112, 286, 107, 278,
         70, 299, 115, 258,  67, 105, 116, 105, 122, 270, 267,  89, 262,  32,
        272, 256,  97, 275,  32, 114, 280, 111, 108, 118, 101, 260, 114,  97,
        257, 263,  32, 283, 100, 105, 256, 257, 266,  32, 283, 102,  97, 109,
        105, 115, 104,  63, 271,  65, 275, 267,  82, 280, 111, 108, 118, 101,
        100,  46,  32, 114, 280, 111, 108, 118, 101, 100, 278,  70, 299, 115,
        258,  67, 105, 116, 105, 122, 270, 267,  70, 299, 296, 261, 281,  32,
        107, 110, 285,  32,  67,  97, 105, 117, 259,  77, 272,  99, 105, 117,
        259, 279, 294, 105, 101, 102,  32, 270, 101, 109, 265, 283, 295, 112,
        101, 111, 112, 108, 101, 278,  65, 275, 267,  87, 256, 1

In [12]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_length, (batch_size,))
    x = torch.stack([data[i:i+context_length] for i in ix])
    y = torch.stack([data[i+1:i+context_length+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPT(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(context_length, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -context_length:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [13]:
model = GPT()
model = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

10.920492 M parameters


In [16]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for epoch in range(max_epochs):

    # every once in a while evaluate the loss on train and val sets
    if epoch % eval_interval == 0 or epoch == max_epochs - 1:
        losses = estimate_loss()
        print(f"step {epoch}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 5.7375, val loss 5.7393
step 50: train loss 3.9162, val loss 3.9069
step 100: train loss 3.3886, val loss 3.3822
step 150: train loss 3.1850, val loss 3.2042
step 200: train loss 3.0185, val loss 3.0725
step 250: train loss 2.8962, val loss 2.9711
step 300: train loss 2.7648, val loss 2.8731
step 350: train loss 2.6156, val loss 2.7682
step 400: train loss 2.5160, val loss 2.7032
step 450: train loss 2.4196, val loss 2.6267
step 500: train loss 2.3551, val loss 2.5874
step 550: train loss 2.2861, val loss 2.5332
step 600: train loss 2.2406, val loss 2.4935
step 650: train loss 2.2110, val loss 2.4799
step 700: train loss 2.1852, val loss 2.4630
step 750: train loss 2.1265, val loss 2.4177
step 800: train loss 2.0860, val loss 2.3976
step 850: train loss 2.0472, val loss 2.3486
step 900: train loss 2.0326, val loss 2.3347
step 950: train loss 2.0145, val loss 2.3309
step 999: train loss 1.9863, val loss 2.3086


In [19]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(tokenizer.decode(model.generate(context, max_new_tokens=100)[0].tolist()))

 IUS:
Well supparte us urged of the were,
Than cilty tooth! nust O Laroce!
You mele? I want hom show to us.

WARCWICK:
Where'll I says e
